In [ ]:
from pathlib import Path
from hallmark.eht_datatree import build_tree
from hallmark.fmt_detection import scan_inventory
import pandas as pd
pd.set_option("display.width", 300)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

a dataree can be built based on the path to the root and the data's fmts. Multiple fmts can be inputted. If no fmts are inputted, the program will try to detect them from the file name patterns, but this functionality still isn't perfect. Flag for if the data is L1 or L2 as the two trees are built slightly different

In [ ]:
datasets = ["EHTC_CenA2017_Jul2021", "EHTC_M87-2018_Mar2024","EHTC_SgrAmwl2017_May2022",
             "EHTC_First3C279Results_May2020", "EHTC_M87mwl2017_Apr2021",      
             "EHTC_metadata2018_Dec2023", "EHTC_FirstM87Results_Apr2019",
             "EHTC_M87mwl2018_Dec2024", "EHTC_FirstSgrAPol_Mar2024",
             "EHTC_M87pol2017_Nov2023", "EHTC_FirstSgrAResults_May2022",
             "EHTC_MonitoringM87_Sep2020"]
root = Path(f"~/{datasets[10]}").expanduser()
tree = build_tree(root, None, data_type="L2")
print("tree keys:", list(tree.keys()))
#files = scan_inventory(root)
#ames = sorted(Path(f).name for f in files)
#for name in names:
#   print(name)

"meta" contains all the housekeeping files, which will be any files besides drives or files that have fmt naming conventions

In [ ]:
print("=== META ===")
print(tree["meta"])

In [ ]:
if "drives" in tree:
    print("=== DRIVES ===")
    print(tree["drives"])

"data" will be a nested dict that has an outer list of all the different fmts

In [ ]:
print("=== DATA FMTS ===")
fmt_keys = list(tree["data"].keys())
for stem in fmt_keys:
    print(f"{stem}")

each fmt has a different Paraframe for each stem, as well as a Paraframe of all the stems merged incase someone wants to view all the data for one fmt

In [ ]:
print(f"=== {fmt_keys[0]} STEMS ===")
stem_keys = list(tree["data"][fmt_keys[0]].keys())
for key in tree["data"][fmt_keys[0]].keys():
    print(key)

files that share fmts besides a missing parameter will also be grouped with that fmt, despite technically being parsed with a seperate fmt.

In [ ]:
print(f"=== ALL PARAFRAME  ===")
print(tree["data"][fmt_keys[0]]["all"])

The invindual stem Paraframes contain the same data with the different file formats

In [ ]:
print(f"=== {stem_keys[0]} STEM ===")
print(
tree["data"][fmt_keys[0]][stem_keys[0]])

The merged paraframe can be filtered to produce the same output as any given stem

In [ ]:
print("=== MERGED PARAFRAME FILTERED ===")
pf = tree["data"][fmt_keys[0]]["all"]

# get cols and params to ease testing and avoid hardcoding values
param_columns = [column for column in pf.columns if column not in ("path", "ext")]
filter_kwargs = {col: pf.iloc[0][col] for col in param_columns}
for column, value in filter_kwargs.items():
    pf = pf(**{column: value})
    
# can also print by doing print(pf(column0="value0")(column1="value1"),...)
print(pf)

An L1 datatree is built similarly to the L2 case. Each zip drive is treated as its own L2 dataset, and prior to that will rely on recursive build_tree calls creating new branches for each folder until the zip drives are hit

In [ ]:
root = Path("~/eht_local_copy").expanduser()
drive_fmts = [
    "e17a10-7-hi-{p0}.{p1}",
    "e17b06-7-hi_{p2}.{p3}",
    "ff-{p0}-{p1}-{p2}-{p3}-{p4}.{p5}",
    "{p0}.B.{p1}.{p2}",]
tree = build_tree(root, drive_fmts, data_type="L1")
root_keys = list(tree.keys())
print("=== ROOT KEYS ===")
print(root_keys)

No "data" branches will be made until we reach a folder that has compressed drives

In [ ]:
all_project_keys = {}
for key in root_keys:
    if key == "meta":
        print("=== META PARAFRAME ===")
        print(tree[key])
        print()
    elif key == "drives":
        print("=== DRIVES PARAFRAME ===")
        print(tree[key])
        print()
    else:
        project_keys = list(tree[key].keys())
        all_project_keys[key] = project_keys
        print(f"=== {key} KEYS ===")
        for key in project_keys:
            print(key)
        print()

In [ ]:
all_extract_keys = {}
for project_name in all_project_keys:
    print(f"=== {project_name} PROJECT ===")
    for extract_name in all_project_keys[project_name]:
        if extract_name == "meta":
            print(f"=== {extract_name} PARAFRAME ===")
            print(tree[project_name][extract_name])
            print()
        elif extract_name == "drives":
            print(f"=== {extract_name} PARAFRAME ===")
            print(tree[project_name][extract_name])
            print()
        else:
            extract_keys = list(tree[project_name][extract_name].keys())
            all_extract_keys[f"{project_name}/{extract_name}"] = extract_keys
            print(f"=== {extract_name} KEYS ===")
            for key in extract_keys:
                print(key)
            print()
    print()

In [ ]:
for project_name in all_project_keys:
    print(f"=== {project_name} PROJECT ===")
    print()
    if "meta" in tree[project_name]:
        print("=== meta PARAFRAME ===")
        print(tree[project_name]["meta"])
        print()
    elif "drives" in tree[project_name]:
        print("=== drives PARAFRAME ===")
        print(tree[project_name]["drives"])
        print()
    for extract_name in all_project_keys[project_name]:
        extract_dict = tree[project_name][extract_name]
        if "data" in extract_dict:
            print(f"=== {extract_name} FMTS ===")
            for fmt in extract_dict["data"]:
                print(f"{fmt}")
            print()
    print()

inconsistant naming conventions makes it difficult to fair all files to the correct fmt

In [ ]:
for project_name in all_project_keys:
    print(f"=== {project_name} PROJECT ===")
    print()
    for extract_name in all_project_keys[project_name]:
        if extract_name != "meta" and extract_name != "drives":
            print(f"=== {extract_name} EXTRACT ===")
            print()
        extract_dict = tree[project_name][extract_name]
        if "data" in extract_dict:
            for fmt_str, stems in extract_dict["data"].items():
                print(f"=== {fmt_str} FMT PARAFRAME ===")
                print(stems["all"])
                print()
    print()